# 08 – Person Details

Esplorazione e data cleaning del dataset `person_details.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona su MyAnimeList |
| `url` | URL della pagina MAL della persona |
| `website_url` | Sito web personale |
| `image_url` | URL dell'immagine del profilo |
| `name` | Nome completo |
| `given_name` | Nome proprio |
| `family_name` | Cognome |
| `birthday` | Data di nascita |
| `favorites` | Numero di utenti che l'hanno tra i preferiti |
| `relevant_location` | Posizione |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [554]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
df_pd = pd.read_csv('../datasets/person_details.csv')
print(f'Shape: {df_pd.shape}')
print()
df_pd.info()
df_pd.head()

Shape: (76699, 10)

<class 'pandas.DataFrame'>
RangeIndex: 76699 entries, 0 to 76698
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   person_mal_id      76699 non-null  int64
 1   url                76699 non-null  str  
 2   website_url        17435 non-null  str  
 3   image_url          76699 non-null  str  
 4   name               76697 non-null  str  
 5   given_name         46446 non-null  str  
 6   family_name        58058 non-null  str  
 7   birthday           17238 non-null  str  
 8   favorites          76699 non-null  int64
 9   relevant_location  76699 non-null  str  
dtypes: int64(2), str(8)
memory usage: 5.9 MB


,person_mal_id,url,website_url,image_url,name,given_name,family_name,birthday,favorites,relevant_location
0,1,https://myanimelist.net/people/1/Tomokazu_Seki,NaN,https://cdn.myanimelist.net/images/voiceactors...,Tomokazu Seki,智一,関,1972-09-08T00:00:00+00:00,6219,"Berlin, Germany"
1,2,https://myanimelist.net/people/2/Tomokazu_Sugita,https://agrs.co.jp/,https://cdn.myanimelist.net/images/voiceactors...,Tomokazu Sugita,智和,杉田,1980-10-11T00:00:00+00:00,47666,"Los Angeles, USA"
2,3,https://myanimelist.net/people/3/Satsuki_Yukino,NaN,https://cdn.myanimelist.net/images/voiceactors...,Satsuki Yukino,さつき,ゆきの,1970-05-25T00:00:00+00:00,1777,"Madrid, Spain"
3,4,https://myanimelist.net/people/4/Aya_Hirano,http://ayahirano.jp/,https://cdn.myanimelist.net/images/voiceactors...,Aya Hirano,綾,平野,1987-10-08T00:00:00+00:00,18374,"Paris, France"
4,5,https://myanimelist.net/people/5/Kenichi_Suzumura,https://intention-k.com,https://cdn.myanimelist.net/images/voiceactors...,Kenichi Suzumura,健一,鈴村,1974-09-12T00:00:00+00:00,5176,"Osaka, Japan"


## 1.2 Rimozione duplicati esatti

Con `relevant_location` rimossa, le righe che prima differivano solo per quella colonna diventano duplicati esatti. Rimuoviamo mantenendo solo la prima occorrenza.

In [555]:
n_originale = len(df_pd)

mask_dup = df_pd.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pd[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pd.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pd):,}')

Righe totali coinvolte in duplicazioni : 6
  → prime occorrenze mantenute         : 3
  → occorrenze extra rimosse           : 3

Righe prima della rimozione : 76,699
Righe dopo la rimozione     : 76,696


## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

ID univoco della persona su MAL. È la **chiave primaria** del dataset. I valori duplicati **non sono attesi**.

In [556]:
analyze(df_pd['person_mal_id'])


  Nome serie                     person_mal_id
  dtype                          int64
  Dimensione                     76,696
  Range indice                   0 … 76698
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   76,696
  Valori non nulli               76,696  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           0  (0.00%)
  Positivi                       76,696   (100.00%)
  Negativi                       0   (0.00%)
  Valori unici                   76,594  (99.87%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            1
  Max                            90,015
  Range                          90,014
  Media                          48,631.73
  Mediana                        51,493.50
  Moda/e                         4,564, 6,638, 7,025
 

**Osservazioni:**
- Nessun null
- Il dtype è già `int64` quindi nessuna conversione necessaria. 
- Si nota che 99.87% dei valori sono univoci. Stampiamo un campione per verificare i duplicati. 

In [557]:
print(f'Null in person_mal_id      : {df_pd["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id : {df_pd["person_mal_id"].duplicated().sum()}')

# Campione righe duplicate
mask_pk_dup = df_pd.duplicated(subset=['person_mal_id'], keep=False)
df_dup = df_pd[mask_pk_dup].sort_values('person_mal_id')
print(f'\nRighe coinvolte: {len(df_dup)} | person_mal_id unici duplicati: {df_dup["person_mal_id"].nunique()}')
df_dup.head(20)

Null in person_mal_id      : 0
Duplicati in person_mal_id : 102

Righe coinvolte: 183 | person_mal_id unici duplicati: 81


,person_mal_id,url,website_url,image_url,name,given_name,family_name,birthday,favorites,relevant_location
4170,4564,https://myanimelist.net/people/4564/Natsuko_Ta...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Natsuko Takahashi,ナツコ,高橋,NaN,19,"Mumbai, India"
4171,4564,https://myanimelist.net/people/4564/Natsuko_Ta...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Natsuko Takahashi,ナツコ,高橋,NaN,19,"Paris, France"
4172,4564,https://myanimelist.net/people/4564/Natsuko_Ta...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Natsuko Takahashi,ナツコ,高橋,NaN,19,"Cape Town, South Africa"
6017,6638,https://myanimelist.net/people/6638/Toshio_Suzuki,NaN,https://cdn.myanimelist.net/images/voiceactors...,Toshio Suzuki,敏夫,鈴木,1948-08-19T00:00:00+00:00,354,"Madrid, Spain"
6018,6638,https://myanimelist.net/people/6638/Toshio_Suzuki,NaN,https://cdn.myanimelist.net/images/voiceactors...,Toshio Suzuki,敏夫,鈴木,1948-08-19T00:00:00+00:00,354,"Paris, France"
6019,6638,https://myanimelist.net/people/6638/Toshio_Suzuki,NaN,https://cdn.myanimelist.net/images/voiceactors...,Toshio Suzuki,敏夫,鈴木,1948-08-19T00:00:00+00:00,354,"Mumbai, India"
6379,7025,https://myanimelist.net/people/7025/Yoshiji_Ki...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Yoshiji Kigami,益治,木上,1957-12-28T00:00:00+00:00,49,"Los Angeles, USA"
6380,7025,https://myanimelist.net/people/7025/Yoshiji_Ki...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Yoshiji Kigami,益治,木上,1957-12-28T00:00:00+00:00,49,"Paris, France"
6381,7025,https://myanimelist.net/people/7025/Yoshiji_Ki...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Yoshiji Kigami,益治,木上,1957-12-28T00:00:00+00:00,49,"Madrid, Spain"
6538,7194,https://myanimelist.net/people/7194/Youta_Tsur...,NaN,https://cdn.myanimelist.net/images/voiceactors...,Youta Tsuruoka,陽太,鶴岡,1959-04-28T00:00:00+00:00,448,"Chicago, USA"


I duplicati consistono in righe in cui l'unico valore che cambia è `relevant_location`. Non possiamo rimuovere le righe in quanto non è possibile sapere quale location è quella corretta. In più, la presenza di questa colonna rompe la struttura della tabella dove ogni persona dovrebbe comparire in una sola riga dove la chiave primaria è   `person_mal_id`. Abbiamo deciso di rimuovere la colonna `relevant_location` e infine rimuovere le righe duplicate mantenendo solo la prima occorrenza. 

In [558]:
# Rimozione colonna 
df_pd.drop(columns=['relevant_location'], inplace=True)

# Rimozione duplicati indotti (ora sono duplicati esatti)
n_prima = len(df_pd)
df_pd.drop_duplicates(subset=['person_mal_id'], keep='first', inplace=True)
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe prima : {n_prima:,}')
print(f'Righe dopo  : {len(df_pd):,}')
print(f'Rimosse     : {n_prima - len(df_pd):,}')

Righe prima : 76,696
Righe dopo  : 76,594
Rimosse     : 102


### 2.2 `url`

URL della pagina MAL della persona. Ci aspettiamo unicità e nessun null. Utilizziamo `strip()` per rimuovere eventuali spazzi vuoti all inizio e fine del URL. Effetuiamo poi l'analisi usando la nostra libreria `dataset_analyzer` e facciamo un controllo sul format del URL. Controlliamo anche la consistenza `person_mal_id` ↔ `url`. L'URL contiene l'ID del personaggio nel percorso (`/people/{id}/`). Verifichiamo che l'ID estratto dall'URL corrisponda al valore di `person_mal_id` per ogni riga.

In [559]:
df_pd['url'] = df_pd['url'].str.strip()
analyze(df_pd['url'])

pattern = r'^https://myanimelist\.net/people/\d+/\S+$'
non_conformi = df_pd[~df_pd['url'].str.match(pattern)]
print(f'URL non conformi al formato atteso: {len(non_conformi)}')
if len(non_conformi) > 0:
    print(non_conformi[['person_mal_id', 'url', 'name']].to_string(index=True))

id_from_url = df_pd['url'].str.extract(r'/people/(\d+)/')[0].astype(int)
mismatch = (id_from_url != df_pd['person_mal_id'])
print(f'\nRighe con ID non coerente tra url e person_mal_id: {mismatch.sum()}')


  Nome serie                     url
  dtype                          str
  Dimensione                     76,594
  Range indice                   0 … 76593
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   76,594
  Valori non nulli               76,594  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   76,594  (100.00%)
  Valori duplicati               0  (0.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  37  caratteri
  Lunghezza max                  84  caratteri
  Lunghezza media                48.91  caratteri
  Lunghezza mediana              49.0000  caratteri
  Dev. standard lunghezza        3.65
  IQR lunghezza                  4.0000

Valori di lunghezza 

**Nessuna pulizia necessaria.**  
Nessun null, tutti gli URL sono unici e rispettano il pattern atteso (coerente con `person_mal_id` come chiave primaria).

### 2.3 `website_url`

Sito web personale della persona, campo **opzionale** su MAL. I null sono attesi in quanto alcune persone potrebbero non aver inserito il proprio sito. I duplicati sono possibili ma rari come per esempio più persone potrebbero condividere un sito.

In [560]:
df_pd['website_url'] = df_pd['website_url'].str.strip()
analyze(df_pd['website_url'])

# 1. Pattern: i valori non-null devono iniziare con http:// o https://
mask_invalid = df_pd['website_url'].notna() & ~df_pd['website_url'].str.match(r'^https?://')
print(f'URL senza protocollo http/https : {mask_invalid.sum()}')
if mask_invalid.sum() > 0:
    print(df_pd.loc[mask_invalid, ['person_mal_id', 'name', 'website_url']].to_string(index=True))

# 2. Valori anomali: presenza di spazi o assenza di punto (probabile non-URL)
mask_anomali = (df_pd['website_url'].notna()
                & (df_pd['website_url'].str.contains(r' ') | ~df_pd['website_url'].str.contains(r'\.')))
print(f'Valori anomali (spazi o senza punto) : {mask_anomali.sum()}')
if mask_anomali.sum() > 0:
    print(df_pd.loc[mask_anomali, ['person_mal_id', 'name', 'website_url']].to_string(index=True))



  Nome serie                     website_url
  dtype                          str
  Dimensione                     76,594
  Range indice                   0 … 76593
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   76,594
  Valori non nulli               17,408  (22.73%)
  Null / NaN                     59,186  (77.27%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   17,332  (22.63%)
  Valori duplicati               76  (0.10%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  9  caratteri
  Lunghezza max                  249  caratteri
  Lunghezza media                32.84  caratteri
  Lunghezza mediana              30.0000  caratteri
  Dev. standard lunghezza        11.44
  IQR lunghezza                  11.0000

Valor

- I null (59.186, ~77%) sono strutturali: la maggior parte delle persone non ha un sito web registrato su MAL.
- Risultano 22 righe con valori anomali che abbiamo deciso di tenere perchè non sono rilevanti per la nostra analisi statistica e per non perdere le informazioni delle altre colonne. 

**Nessuna pulizia necessaria.**  


### 2.4 `image_url`

URL dell'immagine del profilo della persona su MAL. Effetuiamo l'analisi usando la nostra libreria `dataset_analyzer` dopo aver usato `strip()` per rimuovere spazi vuoti e facciamo poi un controllo sul format del URL.

In [561]:
df_pd['image_url'] = df_pd['image_url'].str.strip()
analyze(df_pd['image_url'])

pattern_img = r'^https://cdn\.myanimelist\.net/(images/voiceactors/\d+/\d+\.jpg|img/sp/icon/apple-touch-icon-256\.png)$'
non_conformi_img = df_pd[~df_pd['image_url'].str.match(pattern_img)]
print(f'Immagini non conformi al formato atteso: {len(non_conformi_img)}')
if len(non_conformi_img) > 0:
    print(non_conformi_img[['person_mal_id', 'name', 'image_url']].to_string(index=True))


  Nome serie                     image_url
  dtype                          str
  Dimensione                     76,594
  Range indice                   0 … 76593
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   76,594
  Valori non nulli               76,594  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   28,429  (37.12%)
  Valori duplicati               48,165  (62.88%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  55  caratteri
  Lunghezza max                  64  caratteri
  Lunghezza media                61.75  caratteri
  Lunghezza mediana              64.0000  caratteri
  Dev. standard lunghezza        2.94
  IQR lunghezza                  6.0000

Valori di

**Osservazioni:**  
- Non ci sono stringhe vuote o null. Tutta la colonna è popolata. 
- Ci sono 48.165 URL duplicati (62.88%). Tutti corrispondono a un link che porta a un'immagine placeholder. A ogni persona senza immagine reale viene assegnato il logo MAL. Non è un errore e non è necessaria nessuna pulizia. 
- Ci sono 28.429 URL unici (37.12%). Tutti le persone con immagine reale hanno un URL univoco.
- Tutte le URL seguono il formato corretto senza nessuna anomalia strutturale.
- Nella distribuzione delle lunghezze, le fasce 59–63 hanno 0 occorrenze, il che significa che le URL si dividono in due gruppi: percorso /images/characters/ (55–58 caratteri) e percorso placeholder /img/sp/icon/ (64 caratteri). 

**Nessuna pulizia necessaria**


### 2.5 `name`

Nome completo della persona. I duplicati sono possibili.

In [562]:
df_pd['name'] = df_pd['name'].str.strip()
analyze(df_pd['name'])


  Nome serie                     name
  dtype                          str
  Dimensione                     76,594
  Range indice                   0 … 76593
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   76,594
  Valori non nulli               76,592  (100.00%)
  Null / NaN                     2  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   74,508  (97.28%)
  Valori duplicati               2,084  (2.72%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  47  caratteri
  Lunghezza media                12.06  caratteri
  Lunghezza mediana              12.0000  caratteri
  Dev. standard lunghezza        3.65
  IQR lunghezza                  4.0000

Valori di lunghez

**Osservazione:**  
- Ci sono 2 valori null. Li stampiamo per vedere se ci sono altri dati che possono essere utili. 
- Ci sono 2.084 duplicati. Stampiamo una parte per indagare meglio. 
- Ci sono nomi con un solo carattere, contenenti @ oppure interamente numerici. Stampiamo una parte per verificare se si tratta di errori. 

In [563]:
# Null
null_name = df_pd[df_pd['name'].isna()]
print(f'Righe con name null: {len(null_name)}')
print(null_name.to_string(index=True))

# Duplicati
dup_name = df_pd[df_pd['name'].duplicated(keep=False)].sort_values('name')
print(f'\nRighe con name duplicato: {len(dup_name)}')
print(dup_name.head(20).to_string(index=True))

# Nomi da 1 carattere
singoli = df_pd[df_pd['name'].str.len() == 1]
print(f'\nNomi da 1 carattere: {len(singoli)}')
print(singoli.head(10).to_string(index=True))

# Nomi contenenti @
con_at = df_pd[df_pd['name'].str.contains('@|＠', regex=True, na=False)]
print(f"\nNomi contenenti '@': {len(con_at)}")
print(con_at.head(10).to_string(index=True))

# Nomi solo numerici
solo_num = df_pd[df_pd['name'].str.fullmatch(r'\d+', na=False)]
print(f'\nNomi solo numerici: {len(solo_num)}')
print(solo_num.head(10).to_string(index=True))

Righe con name null: 2
       person_mal_id                                        url website_url                                                         image_url name given_name family_name birthday  favorites
46633          59857  https://myanimelist.net/people/59857/None         NaN  https://cdn.myanimelist.net/img/sp/icon/apple-touch-icon-256.png  NaN        NaN          のね      NaN          0
75815          89106  https://myanimelist.net/people/89106/NULL         NaN  https://cdn.myanimelist.net/img/sp/icon/apple-touch-icon-256.png  NaN        NaN         NaN      NaN          0

Righe con name duplicato: 3805
       person_mal_id                                         url                         website_url                                                         image_url   name given_name family_name                   birthday  favorites
73307          86598     https://myanimelist.net/people/86598/Dd                                 NaN  https://cdn.myanimelist.net/img/sp/ico

- I due valori nulli corrispondono a due profili vuoti che verranno rimossi. 
- Il resto dei valori corriponde a righe che contengono informazioni parzialmente mancanti. Prima di decidere se rimuoverli o meno, verifichiamo se compaiono in altri dataset come foreign keys. Le righe che non vengono mai referenziate, vengono rimosse. 

In [564]:
# Rimozione righe con name null
print(f'Righe prima: {len(df_pd):,}')
df_pd.dropna(subset=['name'], inplace=True)
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe dopo : {len(df_pd):,}')

# Verifica referenze degli ID anomali restanti negli altri dataset
from foreign_key_analyzer import check_pk_referenced

ids_anomali = pd.Series(
    list(
        set(singoli['person_mal_id'])
        | set(con_at['person_mal_id'])
        | set(solo_num['person_mal_id'])
        | set(dup_name['person_mal_id'])
    ),
    name='person_mal_id'
)

df_paw  = pd.read_csv('../datasets/person_anime_works.csv')
df_pvw  = pd.read_csv('../datasets/person_voice_works.csv')
df_pan  = pd.read_csv('../datasets/person_alternate_names.csv')
df_favs = pd.read_csv('../datasets/favs.csv')

mask_unreferenced = check_pk_referenced(
    parent   = ids_anomali,
    children = {
        'person_anime_works'     : df_paw['person_mal_id'],
        'person_voice_works'     : df_pvw['person_mal_id'],
        'person_alternate_names' : df_pan['person_mal_id'],
        'favs (people)'          : df_favs[df_favs['fav_type'] == 'people']['id'],
    },
    parent_df   = df_pd[df_pd['person_mal_id'].isin(ids_anomali)],
    sample_rows = 10
)
# Rimozione degli ID anomali non referenziati in nessuna tabella figlia
ids_da_rimuovere = ids_anomali[mask_unreferenced]
print(f'Righe prima della rimozione: {len(df_pd):,}')
df_pd = df_pd[~df_pd['person_mal_id'].isin(ids_da_rimuovere)]
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe dopo la rimozione    : {len(df_pd):,}')
print(f'Righe rimosse              : {len(ids_da_rimuovere):,}')

Righe prima: 76,594
Righe dopo : 76,592

  Colonna PK  (tabella padre)         person_mal_id
  Tabelle figlie verificate           person_anime_works, person_voice_works, person_alternate_names, favs (people)
────────────────────────────────────────────────────────────────────────────────

Referenze per tabella figlia
────────────────────────────────────────────────────────────────────────────────

    person_anime_works                1,722 / 3,880 ID trovati  (44.4%)
    person_voice_works                557 / 3,880 ID trovati  (14.4%)
    person_alternate_names            545 / 3,880 ID trovati  (14.0%)
    favs (people)                     652 / 3,880 ID trovati  (16.8%)

Riepilogo globale
────────────────────────────────────────────────────────────────────────────────

  PK totali analizzati                3,880
  ✓  Referenziati in almeno 1 figlia  2,497  (64.4%)
  ✗  Non referenziati in nessuna      1,383  (35.6%)

ID non referenziati (rimovibili in sicurezza)
──────────────────

### 2.6 `given_name`

Nome proprio in giapponese. I null sono strutturali: non disponibile per persone non giapponesi o quando non registrato su MAL.

In [565]:
df_pd['given_name'] = df_pd['given_name'].str.strip()
analyze(df_pd['given_name'])


  Nome serie                     given_name
  dtype                          str
  Dimensione                     75,211
  Range indice                   0 … 75210
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   75,211
  Valori non nulli               45,808  (60.91%)
  Null / NaN                     29,403  (39.09%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   22,014  (29.27%)
  Valori duplicati               23,794  (31.64%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  28  caratteri
  Lunghezza media                2.18  caratteri
  Lunghezza mediana              2.0000  caratteri
  Dev. standard lunghezza        0.89
  IQR lunghezza                  1.0000

Valori

**Osservazioni:**  
- I null (~39%) sono strutturali e non richiedono intervento.  
- Dalla distribuzione delle lunghezze (min 1, max 28, media ~2.18) emergono valori anomali:  
  - stringhe con soli caratteri latini (il campo dovrebbe contenere kanji/kana);  
  - stringhe molto lunghe (> 15 car.) che non sembrano nomi propri;  
  - valori con parentesi che mescolano lingue o aggiungono annotazioni.  

Verifichiamo e puliamo.

In [566]:
# Stringhe con soli caratteri latini (attesi kanji/kana)
solo_latin_gn = df_pd[df_pd['given_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)]
print(f'Solo caratteri latini: {len(solo_latin_gn)}')
print(solo_latin_gn[['person_mal_id', 'name', 'given_name']].head(15).to_string(index=True))

# Stringhe molto lunghe (> 15 caratteri)
lunghi_gn = df_pd[df_pd['given_name'].str.len() > 15]
print(f'Lunghezza > 15 caratteri: {len(lunghi_gn)}')
print(lunghi_gn[['person_mal_id', 'name', 'given_name']].head(15).to_string(index=True))

# Valori contenenti parentesi
con_par_gn = df_pd[df_pd['given_name'].str.contains(r'[\(\)\[\]]', na=False)]
print(f'Contenenti parentesi: {len(con_par_gn)}')
print(con_par_gn[['person_mal_id', 'name', 'given_name']].head(15).to_string(index=True))

Solo caratteri latini: 94
      person_mal_id                name                given_name
1273           1296           Joey Hood                      Joey
1342           1367     Peter Fernandez                     Peter
1711           1810        Hynden Walch        Heidi Hynden Walch
1717           1822  Michelle Rodriguez  Mayte Michelle Rodriguez
1720           1828         Dave Mallow           David J. Mallow
1726           1842          Andre Ware          Frank Andre Ware
2085           2238           Aoi Kujou                       AOI
4509           5048            Stan Lee            Stanley Martin
4776           5350  Phillip Wesolowski                   Phillip
5122           5739        Sunjun Hwang      Coolest Guy on Earth
5136           5754       Sound Horizon             Sound Horizon
5190           5816     Ilaria Graziano                         -
5653           6328    Thores Shibamoto                    Thores
6093           6806    Jonathan Diamani           

- Le stringhe con soli caratteri latini e quelle molto lunghe sono profili non giapponesi o voci di fantasia. Il campo `given_name` non è significativo per loro e quindi lo impostiamo a `null`.  
- I valori con parentesi rientrano nelle categorie precedenti (es. trascrizioni multi-lingua), trattati nello stesso modo. 

In [567]:
# Maschera valori anomali in given_name
mask_anomali_gn = (
    df_pd['given_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)   # solo Latin
    | (df_pd['given_name'].str.len() > 15)                                # troppo lunghi
    | df_pd['given_name'].str.contains(r'[\(\)\[\]]', na=False)       # parentesi
)

n_anomali = mask_anomali_gn.sum()
print(f'Valori anomali in given_name: {n_anomali}')

# Azzeriamo a NaN anziché rimuovere la riga: la persona può essere valida
df_pd.loc[mask_anomali_gn, 'given_name'] = np.nan
print(f'Null in given_name dopo pulizia: {df_pd["given_name"].isna().sum():,} ({df_pd["given_name"].isna().mean()*100:.1f}%)')

Valori anomali in given_name: 103
Null in given_name dopo pulizia: 29,506 (39.2%)


### 2.7 `family_name`

Cognome in giapponese (kanji/kana). Analogo a `given_name`, i null sono strutturali per le stesse ragioni.

In [568]:
df_pd['family_name'] = df_pd['family_name'].str.strip()
analyze(df_pd['family_name'])


  Nome serie                     family_name
  dtype                          str
  Dimensione                     75,211
  Range indice                   0 … 75210
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   75,211
  Valori non nulli               56,703  (75.39%)
  Null / NaN                     18,508  (24.61%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   27,273  (36.26%)
  Valori duplicati               29,430  (39.13%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  31  caratteri
  Lunghezza media                2.37  caratteri
  Lunghezza mediana              2.0000  caratteri
  Dev. standard lunghezza        1.29
  IQR lunghezza                  0.0000

Valor

**Osservazioni:**  
- I null (~24%) sono strutturali e non richiedono intervento.  
- Dalla distribuzione delle lunghezze (min 1, max 31, media ~2.37) emergono valori anomali:  
  - stringhe con soli caratteri latini (il campo dovrebbe contenere kanji/kana);  
  - stringhe molto lunghe (> 15 car.) con nomi di band o valori compositi;  
  - valori con parentesi che mescolano lingue o aggiungono annotazioni.  
Verifichiamo e puliamo.

In [569]:
# Stringhe con soli caratteri latini (attesi kanji/kana)
solo_latin_fn = df_pd[df_pd['family_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)]
print(f'Solo caratteri latini: {len(solo_latin_fn)}')
print(solo_latin_fn[['person_mal_id', 'name', 'family_name']].head(15).to_string(index=True))

# Stringhe molto lunghe (> 15 caratteri)
lunghi_fn = df_pd[df_pd['family_name'].str.len() > 15]
print(f'Lunghezza > 15 caratteri: {len(lunghi_fn)}')
print(lunghi_fn[['person_mal_id', 'name', 'family_name']].head(15).to_string(index=True))

# Valori contenenti parentesi
con_par_fn = df_pd[df_pd['family_name'].str.contains(r'[\(\)\[\]]', na=False)]
print(f'Contenenti parentesi: {len(con_par_fn)}')
print(con_par_fn[['person_mal_id', 'name', 'family_name']].head(15).to_string(index=True))

Solo caratteri latini: 1342
      person_mal_id             name   family_name
219             223             Myco          myco
842             857             KENN          KENN
1273           1296        Joey Hood          Hood
1342           1367  Peter Fernandez     Fernandez
1747           1877            CLAMP         CLAMP
1869           2003              ZUN           ZUN
2190           2352              MEE           MEE
2348           2529     CJ Michalski  CJ Michalski
2509           2709            okama         okama
2674           2885              Key           Key
2752           2970        naked ape     naked ape
2811           3040            Meimu         MEIMU
2981           3246              Rem           Rem
3031           3303          Chi-Ran       CHI-RAN
3043           3316             RaTe          RaTe
Lunghezza > 15 caratteri: 34
       person_mal_id                             name                      family_name
7995            8878                  Do

**Osservazioni:**  
- Le stringhe con soli caratteri latini e quelle oltre i 15 caratteri non sono cognomi giapponesi validi quindi li impostiamo a `null`.  
- I valori con parentesi rientrano nelle stesse categorie (valori compositi o multi-lingua), trattati nello stesso modo.

In [570]:
# Maschera valori anomali in family_name
mask_anomali_fn = (
    df_pd['family_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)   # solo Latin
    | (df_pd['family_name'].str.len() > 15)                              # troppo lunghi
    | df_pd['family_name'].str.contains(r'[\(\)\[\]]', na=False)        # parentesi
)

n_anomali_fn = mask_anomali_fn.sum()
print(f'Valori anomali in family_name: {n_anomali_fn}')

df_pd.loc[mask_anomali_fn, 'family_name'] = np.nan
print(f'Null in family_name dopo pulizia: {df_pd["family_name"].isna().sum():,} ({df_pd["family_name"].isna().mean()*100:.1f}%)')

Valori anomali in family_name: 1366
Null in family_name dopo pulizia: 19,874 (26.4%)


### 2.8 `birthday`

Data di nascita come stringa ISO 8601 con timezone UTC, campo **opzionale**. I null sono strutturali: molte persone non rendono pubblica la data di nascita su MAL. Richiede conversione a `datetime64` e verifica di date implausibili.

In [571]:
df_pd['birthday'] = df_pd['birthday'].str.strip()
analyze(df_pd['birthday'])


  Nome serie                     birthday
  dtype                          str
  Dimensione                     75,211
  Range indice                   0 … 75210
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   75,211
  Valori non nulli               17,154  (22.81%)
  Null / NaN                     58,057  (77.19%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   12,348  (16.42%)
  Valori duplicati               4,806  (6.39%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  25  caratteri
  Lunghezza max                  25  caratteri
  Lunghezza media                25.00  caratteri
  Lunghezza mediana              25.0000  caratteri
  Dev. standard lunghezza        0.00
  IQR lunghezza                  0.0000

Valori 

**Osservazioni:**  
- I valori sono stringhe che convertiamo a `datetime64`.
- I null (~78%) sono strutturali: date di nascita non pubbliche o non registrate su MAL.
- Controlliamo il range per verificare possibili date errate che verranno impostate a `null`.

In [572]:
df_pd['birthday'] = pd.to_datetime(df_pd['birthday'], errors='coerce', utc=True)
print(f'Range          : {df_pd["birthday"].min()} → {df_pd["birthday"].max()}')
print()

# Date implausibili: anno < 1800
mask_passato = df_pd['birthday'].dt.year < 1800
n_passato = mask_passato.sum()
print(f'Date con anno < 1800: {n_passato}')
print(df_pd.loc[mask_passato, ['person_mal_id', 'name', 'url', 'birthday']].head(30).to_string(index=True))

# Date recenti
mask_futuro = df_pd['birthday'].dt.year > 2020
n_futuro = mask_futuro.sum()
print(f'\nDate recenti: {n_futuro}')
print(df_pd.loc[mask_futuro, ['person_mal_id', 'name', 'url', 'birthday']].head(30).to_string(index=True))

# Rimozione di tutti i birthday il cui anno non inizia con 1 o 2 (fuori dal range 1000-2026)
mask_bad_date = df_pd['birthday'].notna() & ~df_pd['birthday'].dt.year.between(1000, 2026)
df_pd.loc[mask_bad_date, 'birthday'] = pd.NaT
print(f'\nDate anomale impostate a NaT : {mask_bad_date.sum()}')
print(f'birthday dtype : {df_pd["birthday"].dtype}')
print(f'Null residui   : {df_pd["birthday"].isna().sum():,}')

Range          : 0178-05-01 00:00:00+00:00 → 2023-06-15 00:00:00+00:00

Date con anno < 1800: 27
       person_mal_id                              name                                                                   url                  birthday
2888            3134               William Shakespeare               https://myanimelist.net/people/3134/William_Shakespeare 1564-04-26 00:00:00+00:00
4057            4482        Johann Wolfgang von Goethe        https://myanimelist.net/people/4482/Johann_Wolfgang_von_Goethe 1749-08-28 00:00:00+00:00
8145            9045                  Charles Perrault                  https://myanimelist.net/people/9045/Charles_Perrault 1628-01-12 00:00:00+00:00
8247            9159             Johann Sebastian Bach             https://myanimelist.net/people/9159/Johann_Sebastian_Bach 1685-03-31 00:00:00+00:00
8248            9160              Ludwig van Beethoven              https://myanimelist.net/people/9160/Ludwig_van_Beethoven 1770-12-16 00:00:00+00:

Abbiamo effettuato un'analisi delle date estreme fuori dal range 1800 - 2020. Dalle date emerse prima del 1800, si nota che si tratta principalmente di autori o compositori classici mentre i valori dopo il 2020 corrispondono a AI oppure gruppi musicali formati recentemente. Abbiamo impostato a null solo gli anni che corrispondevano a valori evidentemente errati come `0800`, `0447`, `0296`, `0178`, etc. 

### 2.9 `favorites`

Numero di utenti MAL che hanno aggiunto questa persona ai preferiti. Valore numerico non negativo, nessun null atteso. 

In [573]:
analyze(df_pd['favorites'])


  Nome serie                     favorites
  dtype                          int64
  Dimensione                     75,211
  Range indice                   0 … 75210
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   75,211
  Valori non nulli               75,211  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           50,936  (67.72%)
  Positivi                       24,275   (32.28%)
  Negativi                       0   (0.00%)
  Valori unici                   1,168  (1.55%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            108,157
  Range                          108,157
  Media                          54.5776
  Mediana                        0.0000
  Moda/e                         0
  Dev. standard        

Nessun null, dtype già `int64`. La distribuzione è fortemente asimmetrica (poche persone molto famose con molti preferiti).

**Nessuna pulizia necessaria.**  

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [580]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_pd):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_pd):>10,}')
print() 

df_pd.to_csv('../datasets_cleaned/person_details_clean.csv', index=False)
print('Salvato: datasets_cleaned/person_details_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :     76,699
Righe dopo cleaning  :     75,211
Righe rimosse totali :      1,488

Salvato: datasets_cleaned/person_details_clean.csv
